# FineWeb-Edu `sample-10BT` → fixed-length corpus (EDA)

Self-contained dev notebook. Streams `HuggingFaceFW/fineweb-edu` (`sample-10BT`), tokenizes with the model specified in `CorpusRegressionConfig.pretrained_tokenizer_model_name`, filters to docs of at least `sample_length` tokens, and takes the first-`sample_length`-token **prefix** of each as a sample. Collects `2 * num_samples`, then shuffles and splits into train/val shards.

Packing is intentionally avoided so each sample corresponds to exactly one document prefix — clean per-sample semantics for a regression target. We're happy with the length bias this induces.

In [ ]:
from __future__ import annotations

import itertools

import plotly.express as px
import polars as pl
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from src import get_repo_base
from src.config.base import BaseConfig

dataset = "HuggingFaceFW/fineweb-edu"
dataset_config = "sample-10BT"
eda_sample = 5_000  # rows pulled for distribution plots; separate from the collected corpus
seed = 42

In [ ]:
class CorpusRegressionConfig(BaseConfig):
    sample_length: int  # canonical 128
    num_samples: int  # canonical 50_000 (per split; train and val each get this many)
    pretrained_tokenizer_model_name: str  # canonical 'HuggingFaceTB/SmolLM2-135M'

    num_regression_bins: int  # Don't worry about this for now. Instantiate to 8


cfg = CorpusRegressionConfig(
    sample_length=128,
    num_samples=50_000,
    pretrained_tokenizer_model_name="HuggingFaceTB/SmolLM2-135M",
    num_regression_bins=8,
)

tokenizer_slug = cfg.pretrained_tokenizer_model_name.split("/")[-1].lower().replace("-", "_")
out_dir = get_repo_base() / "artifacts" / "corpus-regression" / (
    f"fineweb_edu_{tokenizer_slug}_{cfg.num_samples}_x_{cfg.sample_length}"
)
out_dir.mkdir(parents=True, exist_ok=True)
out_train = out_dir / "train.parquet"
out_val = out_dir / "val.parquet"
cfg, out_train, out_val

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.pretrained_tokenizer_model_name)
assert tok.eos_token_id is not None
eos = tok.eos_token_id

print(f"tokenizer:       {cfg.pretrained_tokenizer_model_name}")
print(f"tok.vocab_size:  {tok.vocab_size}")
print(f"len(tok):        {len(tok)}  (includes added/special tokens)")
print(f"eos_token / id:  {tok.eos_token!r} / {eos}")

## 1. Stream the dataset

`streaming=True` avoids the full ~28 GB download. Shuffle with a buffer before taking anything so the EDA sample isn't just the first shard.

In [ ]:
ds_stream = load_dataset(dataset, name=dataset_config, split="train", streaming=True)
ds_stream = ds_stream.shuffle(seed=seed, buffer_size=10_000)
ds_stream

In [ ]:
first = next(iter(ds_stream))
print("fields:", list(first.keys()))
for k, v in first.items():
    s = repr(v)
    print(f"  {k:>14}: {s[:120]}{' ...' if len(s) > 120 else ''}")

## 2. Pull an EDA sample

Take `eda_sample` rows into memory for quick polars/plotly analysis. We retokenize each doc with our tokenizer so the reported lengths match the filter criterion used downstream — the dataset's native `token_count` is GPT-2-based and would misrepresent the filter threshold for other tokenizers.

In [ ]:
rows = list(itertools.islice(ds_stream, eda_sample))
texts = [r["text"] for r in rows]
tok_counts = [
    len(tok.encode(t, add_special_tokens=False))
    for t in tqdm(texts, desc="retokenizing EDA sample")
]
eda = pl.DataFrame({
    "text": texts,
    "token_count": tok_counts,
    "gpt2_token_count": [r["token_count"] for r in rows],
    "language": [r.get("language") for r in rows],
    "score": [r.get("score") for r in rows],
    "url": [r.get("url") for r in rows],
})
eda = eda.with_columns(char_len=pl.col("text").str.len_chars())
eda.select("token_count", "gpt2_token_count", "char_len", "score").describe()

In [ ]:
fig = px.histogram(
    {"token_count": eda["token_count"].to_list()},
    x="token_count",
    nbins=80,
    log_x=True,
    title=(
        f"FineWeb-Edu {dataset_config}: token_count ({eda_sample} docs, "
        f"{cfg.pretrained_tokenizer_model_name})"
    ),
)
fig.add_vline(
    x=cfg.sample_length,
    line_dash="dash",
    annotation_text=f"sample_length={cfg.sample_length}",
)
fig.show()

In [ ]:
q = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
tc = eda["token_count"]
print(f"token_count quantiles ({cfg.pretrained_tokenizer_model_name}):")
for qi in q:
    print(f"  p{int(qi * 100):>2}: {tc.quantile(qi):>8.0f}")
keep = (tc >= cfg.sample_length).sum() / len(tc)
print(f"\n{keep:.1%} of docs have >= {cfg.sample_length} tokens — these are the candidates we keep.")

## 3. Eyeball raw examples

One short doc, one medium, one long — so you can see the range of content that survives the filter.

In [ ]:
picks = (
    eda.sort("token_count")
    .with_row_index()
    .filter(pl.col("index").is_in([len(eda) // 20, len(eda) // 2, len(eda) - len(eda) // 20]))
)
for row in picks.iter_rows(named=True):
    kept = row["token_count"] >= cfg.sample_length
    print(f"--- token_count={row['token_count']}  score={row['score']}  kept={kept} ---")
    print(row["text"][:600].rstrip() + (" …" if len(row["text"]) > 600 else ""))
    print()

## 4. Filter + truncate to document prefixes

Stream, tokenize with `cfg.pretrained_tokenizer_model_name`, skip docs shorter than `cfg.sample_length`, take the first `cfg.sample_length` tokens of each kept doc. Collect `2 * cfg.num_samples` samples (train + val). Re-streams from scratch so the EDA sample above doesn't bias the final corpus.

In [ ]:
target_total = 2 * cfg.num_samples

ds_collect = load_dataset(dataset, name=dataset_config, split="train", streaming=True).shuffle(
    seed=seed, buffer_size=10_000
)

samples: list[list[int]] = []
docs_seen = 0
docs_kept = 0
bar = tqdm(total=target_total, desc=f"collecting {cfg.sample_length}-token prefixes")

for ex in ds_collect:
    text = ex["text"].strip()
    if not text:
        continue
    docs_seen += 1
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) < cfg.sample_length:
        continue
    samples.append(ids[: cfg.sample_length])
    docs_kept += 1
    bar.update(1)
    if docs_kept >= target_total:
        break
bar.close()

keep_rate = docs_kept / max(docs_seen, 1)
print(f"kept {docs_kept} / {docs_seen} docs ({keep_rate:.1%} pass rate)")

## 5. Shuffle + shard into train/val

Docs arrive already shuffled (streaming buffer), but shuffle again for good measure so train/val are interchangeable. `polars.DataFrame.sample(fraction=1.0, shuffle=True, seed=...)` is deterministic given `seed`.

In [ ]:
df = pl.DataFrame({"input_ids": samples}).sample(fraction=1.0, shuffle=True, seed=seed)
train_df = df.head(cfg.num_samples)
val_df = df.slice(cfg.num_samples, cfg.num_samples)
assert len(train_df) == cfg.num_samples and len(val_df) == cfg.num_samples
print(f"train: {len(train_df)}  val: {len(val_df)}  (sample_length={cfg.sample_length})")
train_df.head(2)

## 6. Inspect train samples

Decode a few prefixes back to text as a sanity check.

In [ ]:
for i in (0, 1, 2, len(train_df) // 2, len(train_df) - 1):
    ids = train_df["input_ids"][i].to_list()
    print(f"--- train[{i}]  len={len(ids)} ---")
    print(tok.decode(ids))
    print()

## 7. Persist

Two parquet shards under `out_dir`. Each row is one `input_ids` list of length `cfg.sample_length`.

In [ ]:
train_df.write_parquet(out_train)
val_df.write_parquet(out_val)
for p in (out_train, out_val):
    print(f"wrote {p}  ({p.stat().st_size / 1e6:.1f} MB)")